# P04 — Control de fuga por paráfrasis
## (a) Paso del protocolo y objetivo
**Paso P04**: evitar la **fuga** (*leakage*): que una paráfrasis de una frase de prueba esté en el entrenamiento, lo que infla las métricas. Se controla agrupando las frases por `base_phrase_id` y verificando que **ningún grupo** esté en dos particiones; en los lotes reales se verifica además que no haya **texto idéntico** entre entrenamiento y test ni participantes compartidos. Sustenta el **OE2**: sin este control, el F1 no mide generalización.
**Origen:** demostraciones con el corpus **Sintético**; los controles de los lotes **Reales** se muestran como conteos (hashes de texto, sin frases).

## (b) Código del repositorio que lo implementa
`audit_corpus.py` (casi-duplicados que se fusionan en un mismo grupo) y `split_corpus.py` / `split_corpus_v3.py` / `split_lote2.py` (asignan grupos completos a una partición y verifican «0 solapamientos»). Entradas: `corpus/corpus_metadata.csv`; salidas: `corpus/dataset_split.csv` y el control T05 en la salida del script.

In [1]:
# ----- Preparación mínima (igual en todos los cuadernos) -----
import os, sys, zipfile
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CARPETA_DRIVE = Path("/content/drive/MyDrive/MPSR_colab")
    BASE = Path("/content/mpsr")
else:
    CARPETA_DRIVE = Path(os.environ.get("MPSR_DRIVE", ".")).resolve()
    BASE = Path(os.environ.get("MPSR_BASE", "mpsr_trabajo")).resolve()
BASE.mkdir(parents=True, exist_ok=True)
ZIP_PUBLICO = CARPETA_DRIVE / "MPSR_colab_publico.zip"
if not ZIP_PUBLICO.exists():
    raise SystemExit(f"Falta {ZIP_PUBLICO}. Sube MPSR_colab_publico.zip a la carpeta MPSR_colab de tu Drive (ver LEEME_colab_por_paso.md).")
if not (BASE / "scripts" / "colab_util.py").exists():          # solo descomprime la primera vez
    with zipfile.ZipFile(ZIP_PUBLICO) as z:
        z.extractall(BASE)
sys.path.insert(0, str(BASE / "scripts"))
import colab_util as U
U.iniciar(BASE, CARPETA_DRIVE)                                  # rutas, paquete privado (si existe) y detección de Rasa
from colab_util import *                                        # rotulo, comparar, sha256, leer_json, tabla_scripts, ic_bootstrap, ...

Listo · BASE = base · paquete privado: SÍ · Rasa: no instalado (no hace falta)


In [2]:
tabla_scripts(scripts_de("04"))

▌ORIGEN: Código del proyecto (no es dato del estudio)


,script,paso del protocolo,propósito (docstring),entradas,salidas,cómo ejecutarlo,usa Rasa,rol en este cuaderno
0,audit_corpus.py,"P03, P04",P03 / P04 — Auditoría del Corpus MPSR-Bot y control de fuga por paráfrasis.,corpus/corpus_metadata.csv,"logs/audit_report.txt, corpus/corpus_audit.csv, corpus_summary.json",python scripts/audit_corpus.py # solo reporta,No,Lógica recalculada (etapa 4)


## (c) Celdas de código
### 1. Ningún grupo de paráfrasis en dos particiones (Sintético)

In [3]:
cm = pd.read_csv(BASE / "corpus/corpus_metadata.csv", dtype=str, keep_default_na=False, encoding="utf-8")
sumv3 = leer_json("corpus/corpus_summary.json")
en_varias = int((cm.groupby("base_phrase_id")["split"].nunique() > 1).sum())
rotulo(ORIGEN_SINTETICO, "control de fuga por grupo de paráfrasis (corpus v3)")
print("Grupos de paráfrasis:", cm["base_phrase_id"].nunique(), "| presentes en más de una partición:", en_varias, "| particiones:", cm["split"].value_counts().to_dict())
comparar("P04", "fugas por grupo de paráfrasis", en_varias, sumv3["leaks_detected"], "corpus/corpus_summary.json", tol=0)

▌ORIGEN: Sintético — control de fuga por grupo de paráfrasis (corpus v3)
Grupos de paráfrasis: 236 | presentes en más de una partición: 0 | particiones: {'train': 546, 'test': 81, 'validation': 81}


### 2. Por qué se agrupa: partición por grupos frente a partición al azar (demostración, Sintético)
Para cada frase de validación/prueba se busca su **vecino más parecido en el entrenamiento** (coseno de n-gramas de caracteres) y se cuenta cuántas veces ese vecino es una **paráfrasis de su mismo grupo** (`base_phrase_id`): eso es fuga. Con la partición por grupos debe ser 0; si se reparten frases sueltas al azar, aparece. *Esta demostración no cambia la partición oficial.*

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)).fit(cm["text"].str.lower())
X = vec.transform(cm["text"].str.lower())
grupo = cm["base_phrase_id"].values


def vecino_mismo_grupo(idx_train, idx_eval):
    sim = cosine_similarity(X[idx_eval], X[idx_train])
    vecino = idx_train[sim.argmax(axis=1)]
    return int((grupo[vecino] == grupo[idx_eval]).sum()), float(sim.max(axis=1).mean())


oficial_train = np.where(cm["split"] == "train")[0]
oficial_eval = np.where(cm["split"] != "train")[0]
perm = np.random.default_rng(42).permutation(len(cm))
n_ev = len(oficial_eval)
azar_eval, azar_train = np.sort(perm[:n_ev]), np.sort(perm[n_ev:])
f_of, s_of = vecino_mismo_grupo(oficial_train, oficial_eval)
f_az, s_az = vecino_mismo_grupo(azar_train, azar_eval)
rotulo(ORIGEN_SINTETICO, "DEMOSTRACIÓN — ¿el vecino más parecido del entrenamiento es una paráfrasis del mismo grupo?")
display(pd.DataFrame({"partición": ["por grupos (oficial)", "al azar (frases sueltas)"], "frases evaluadas": [n_ev, n_ev], "vecino del mismo grupo (fuga)": [f_of, f_az], "% con fuga": [round(100 * f_of / n_ev, 1), round(100 * f_az / n_ev, 1)], "similitud media al vecino": [round(s_of, 3), round(s_az, 3)]}))
comparar("P04", "demostración: fuga con la partición por grupos", f_of, 0, "construcción por grupos (P04)", tol=0)
print("Con la partición al azar hay fuga:", f_az > 0, "→ por eso la partición oficial asigna GRUPOS completos a cada partición.")

▌ORIGEN: Sintético — DEMOSTRACIÓN — ¿el vecino más parecido del entrenamiento es una paráfrasis del mismo grupo?


,partición,frases evaluadas,vecino del mismo grupo (fuga),% con fuga,similitud media al vecino
0,por grupos (oficial),162,0,0.0,0.614
1,al azar (frases sueltas),162,10,6.2,0.588


### 3. El antecedente: la validación cruzada sin agrupar daba un F1 inflado (Real/guardado)
En P11.1 la validación cruzada nativa de Rasa **no respetaba** `base_phrase_id`; dio F1 0,778, inflado por fuga. La versión agrupada dio un valor más bajo.

In [5]:
rotulo(ORIGEN_SINTETICO, "texto guardado de evidencias/README.md y p11_1_pruebas/README.md (no se recalcula: requiere Rasa)")
for ruta in ("evidencias/README.md", "evidencias/p11_1_pruebas/README.md"):
    for linea in (BASE / ruta).read_text(encoding="utf-8").splitlines():
        if re.search(r"inflad|CV agrupada|0\.778", linea):
            print("-", linea.strip()[:300])
            break

▌ORIGEN: Sintético — texto guardado de evidencias/README.md y p11_1_pruebas/README.md (no se recalcula: requiere Rasa)
- | [`v3_corpus708/`](v3_corpus708/README.md) | v3: 708 utterances (+60 frases coloquiales en 9 intenciones); re-evaluación de P11.1 | 2026-10-02 | 0.6480 / 0.5786 / 0.6241 ± 0.0286 en el test; CV agrupada DIET 0.655; **no cumple F1 ≥ 0.75**, ver [REPORTE_REEVALUACION](v3_corpus708/REPORTE_REEVALUACIO
- | 03 | `python -m rasa test nlu --nlu data/nlu_full.yml --config configs/rasa_config.yml --cross-validation` | F1 0.778 ± 0.028, exactitud 0.789 ± 0.027 (5 folds, 19 min) | [txt](salidas/03_rasa_test_nlu_crossval.txt) | [png](capturas/03_rasa_test_nlu_crossval.png) |


### 4. Controles en los lotes reales (Real, solo conteos)

In [6]:
# ----- 4b. Lotes REALES: conteos y controles (sin frases) -----
ag = leer_json("recursos/agregados_reales.json")
rotulo(ORIGEN_REAL, "lote 1 (partición v3) y lote 2 — solo conteos")
p1, p2 = ag["particion_lote1"], ag["particion_lote2"]
print("Lote 1 — partición v3:", p1["split"], "| por origen y partición:", p1["origen_por_split"])
print("Lote 2 — partición:", p2["split"], "| entrenamiento por origen:", p2["entrenamiento_por_origen"])
print("Lote 2 — participantes en el test:", p2["participantes_test"], "| situaciones:", p2["situaciones_test"], "| frases por intención (mín–máx):", p2["test_por_intencion_min_max"])

if HAY_PRIVADO:
    rotulo(ORIGEN_REAL, "recalculado desde privado/particion_sin_texto.csv y privado/hash_textos.csv (sin texto)")
    pt = pd.read_csv(PRIV / "particion_sin_texto.csv", dtype=str, keep_default_na=False, encoding="utf-8")
    ht = pd.read_csv(PRIV / "hash_textos.csv", dtype=str, keep_default_na=False, encoding="utf-8")
    l2 = pt[pt["lote"] == "lote2"]
    train2, test2 = l2[l2["split"] == "train"], l2[l2["split"] == "test"]
    print("Lote 2 recalculado — train:", len(train2), "| test:", len(test2), "| excluidas:", int((l2["split"] == "excluida").sum()))
    print("Entrenamiento por origen:", train2["source"].str.replace(r"\s*\(.*", "", regex=True).value_counts().to_dict(), "→ detalle:", train2["source"].value_counts().to_dict())
    h2 = ht[ht["lote"] == "lote2"]
    solape_texto = len(set(h2[h2["split"] == "train"]["sha256"]) & set(h2[h2["split"] == "test"]["sha256"]))
    part_train = set(train2.loc[train2["participant_code"] != "", "participant_code"])
    part_test = set(test2["participant_code"])
    solape_part = len(part_train & part_test)
    l1 = pt[pt["lote"] == "lote1"]
    gp = l1[l1["split"].isin(["train", "validation", "test"])].groupby("base_phrase_id")["split"].nunique()
    solape_grupos = int((gp > 1).sum())
    print("Solapamiento de TEXTO EXACTO entre entrenamiento y test del lote 2 (hash del texto normalizado):", solape_texto)
    print("Participantes presentes a la vez en el entrenamiento real y en el test del lote 2:", solape_part)
    print("Grupos de paráfrasis del lote 1 en más de una partición:", solape_grupos)
    comparar("P04/P05", "lote 2: frases de test", len(test2), 267, "Protocolo V1.8 / particion_lote2_cifras.md", tol=0)
    comparar("P04/P05", "lote 2: frases de entrenamiento", len(train2), 943, "Protocolo V1.8 (707 + 185 + 51)", tol=0)
    comparar("P04/P05", "lote 2: frases excluidas por idénticas al entrenamiento", int((l2["split"] == "excluida").sum()), 13, "Protocolo V1.8 (13 de 280)", tol=0)
    comparar("P04/P05", "lote 2: solapamientos de texto exacto test/entrenamiento", solape_texto, 0, "Protocolo V1.8 («0 solapamientos»)", tol=0)
    comparar("P04/P05", "lote 2: participantes compartidos entrenamiento/test", solape_part, 0, "diseño del lote 2 (solo test, otros participantes)", tol=0)
    ex = pd.read_csv(PRIV / "exclusion_entrenamiento_lote2.csv", dtype=str, keep_default_na=False, encoding="utf-8")
    print("Exclusión por coincidencia exacta:", len(ex), "de 280 recibidas | por fuente:", ex["fuente"].value_counts().to_dict())
else:
    requiere_privado("recálculo de las particiones reales y de los controles de solapamiento")
    c = ag["controles"]
    print("Valores guardados:", c)
    comparar("P04/P05", "lote 2: frases de test (guardado)", p2["split"]["test"], 267, "Protocolo V1.8", tol=0)
    comparar("P04/P05", "lote 2: frases de entrenamiento (guardado)", p2["split"]["train"], 943, "Protocolo V1.8", tol=0)
    comparar("P04/P05", "lote 2: excluidas por idénticas al entrenamiento (guardado)", p2["split"]["excluida"], 13, "Protocolo V1.8", tol=0)
    comparar("P04/P05", "lote 2: solapamiento de texto exacto test/entrenamiento (guardado)", c["lote2_solape_texto_exacto_test_vs_entrenamiento"], 0, "Protocolo V1.8", tol=0)

▌ORIGEN: Real — lote 1 (partición v3) y lote 2 — solo conteos
Lote 1 — partición v3: {'train': 707, 'test': 114, 'validation': 71, 'excluida': 2} | por origen y partición: {'excluida': {'sintético': 1, 'lenguaje real (lote 1)': 1}, 'test': {'lenguaje real (lote 1)': 114}, 'train': {'sintético': 707}, 'validation': {'lenguaje real (lote 1)': 71}}
Lote 2 — partición: {'train': 943, 'test': 267, 'excluida': 13} | entrenamiento por origen: {'sintético': 707, 'lenguaje real (lote 1)': 185, 'sintético (refinamiento ciclo 1)': 51}
Lote 2 — participantes en el test: 25 | situaciones: 56 | frases por intención (mín–máx): [4, 15]
▌ORIGEN: Real — recalculado desde privado/particion_sin_texto.csv y privado/hash_textos.csv (sin texto)
Lote 2 recalculado — train: 943 | test: 267 | excluidas: 13
Entrenamiento por origen: {'sintético': 758, 'lenguaje real': 185} → detalle: {'sintético': 707, 'lenguaje real (lote 1)': 185, 'sintético (refinamiento ciclo 1)': 51}
Solapamiento de TEXTO EXACTO entre entre

In [7]:
cierre("P04 Fuga por paráfrasis")

P04 Fuga por paráfrasis — recalculado frente a reportado: 7 cifras · coinciden: 7 · NO coinciden: 0


,paso,cifra,recalculado,reportado,fuente de lo reportado,coincide
0,P04,fugas por grupo de paráfrasis,0,0,corpus/corpus_summary.json,Sí
1,P04,demostración: fuga con la partición por grupos,0,0,construcción por grupos (P04),Sí
2,P04/P05,lote 2: frases de test,267,267,Protocolo V1.8 / particion_lote2_cifras.md,Sí
3,P04/P05,lote 2: frases de entrenamiento,943,943,Protocolo V1.8 (707 + 185 + 51),Sí
4,P04/P05,lote 2: frases excluidas por idénticas al entrenamiento,13,13,Protocolo V1.8 (13 de 280),Sí
5,P04/P05,lote 2: solapamientos de texto exacto test/entrenamiento,0,0,Protocolo V1.8 («0 solapamientos»),Sí
6,P04/P05,lote 2: participantes compartidos entrenamiento/test,0,0,"diseño del lote 2 (solo test, otros participantes)",Sí


## (d) Cómo leer el resultado
- **0 grupos** del corpus sintético aparecen en dos particiones: el control T05 se cumple. En la demostración, la partición por grupos deja 0 frases cuyo vecino más parecido sea una paráfrasis de su mismo grupo, mientras que repartir frases sueltas al azar sí produce fuga; por eso se agrupa.
- En el **lote 2** los controles son: 0 textos idénticos entre entrenamiento y test (tras excluir 13 de 280 por coincidencia exacta normalizada, nunca por lo que el modelo prediga) y 0 participantes compartidos entre el entrenamiento real y el test.

## Limitaciones
- La similitud por n-gramas detecta parecido de forma, no de significado; en este corpus las paráfrasis de un mismo grupo no son necesariamente más parecidas entre sí que las de otros grupos de la misma intención, por eso el indicador es la pertenencia al mismo grupo del vecino.
- Los 25 participantes del lote 2 resuelven las **mismas 56 situaciones** que el lote 1: no hay fuga de frases ni de personas, pero sí de *situaciones*; por eso el F1 del lote 2 no prueba generalización a situaciones nuevas.